# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/0mneeha93/ML-Track/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

In [6]:
from google.colab import userdata
hf_token = userdata.get('HF_TOKEN')

!pip install -q duckdb huggingface_hub
import duckdb
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf_token (TYPE HUGGINGFACE, TOKEN '{hf_token}');")

import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

print("DuckDB ready.")

DuckDB ready.


## 2. My model under an honest split (before/after)

The out of time test (KMeans fit on February, applied without refitting to March) shows the same four archetypes reappear with closely matching profiles position, impressions, and CTR are all within a similar range for each cluster, and page counts are comparable (the largest shift is the "High Traffic Workhorses" cluster, which grew from 2,776 to 4,020 pages). This is a materially stronger test than the Week 5 stability check, which only resampled rows within the same March window and couldn't have caught drift across time. Observed result: the clustering structure appears directionally stable out of time, not just stable under resampling though one month of out o f time validation is a limited window, and this should be treated as a decision support signal rather than proof of long term stability.

In [7]:
features_feb = con.execute("""
    SELECT
        f.content_hash_id,
        AVG(f.gsc_avg_position) as avg_position,
        SUM(f.gsc_impressions) as impressions_total,
        CASE WHEN SUM(f.gsc_impressions) > 0 THEN SUM(f.gsc_clicks)*1.0/SUM(f.gsc_impressions) ELSE 0 END as ctr,
        c.word_count
    FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-02/*.parquet' f
    JOIN 'hf://datasets/FlyRank/internship-warehouse/dim_content.parquet' c
      ON f.content_hash_id = c.content_hash_id
    WHERE f.gsc_avg_position > 0
    GROUP BY f.content_hash_id, c.word_count
""").df()

features_feb["word_count"] = features_feb["word_count"].fillna(features_feb["word_count"].median())
print("Feb rows:", len(features_feb))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feb rows: 151956


In [10]:
features_df = con.execute("""
    SELECT
        f.content_hash_id,
        AVG(f.gsc_avg_position) as avg_position,
        SUM(f.gsc_impressions) as impressions_total,
        CASE WHEN SUM(f.gsc_impressions) > 0 THEN SUM(f.gsc_clicks)*1.0/SUM(f.gsc_impressions) ELSE 0 END as ctr,
        c.word_count
    FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet' f
    JOIN 'hf://datasets/FlyRank/internship-warehouse/dim_content.parquet' c
      ON f.content_hash_id = c.content_hash_id
    WHERE f.gsc_avg_position > 0
    GROUP BY f.content_hash_id, c.word_count
""").df()

features_df["word_count"] = features_df["word_count"].fillna(features_df["word_count"].median())
print("March rows:", len(features_df))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

March rows: 175304


In [8]:
X_feb = features_feb[["avg_position", "impressions_total", "ctr", "word_count"]]
scaler_feb = StandardScaler().fit(X_feb)
X_feb_scaled = scaler_feb.transform(X_feb)

km_feb = KMeans(n_clusters=4, random_state=42, n_init=10).fit(X_feb_scaled)
feb_centers = pd.DataFrame(km_feb.cluster_centers_, columns=["avg_position","impressions_total","ctr","word_count"])
print("Feb-fit-on-Feb centers:")
print(feb_centers.round(3))

Feb-fit-on-Feb centers:
   avg_position  impressions_total     ctr  word_count
0        -0.412              5.975  -0.031       0.117
1        -0.304             -0.066  -0.031       0.025
2         2.199             -0.231  -0.085      -0.175
3        -0.226             -0.291  19.486      -1.067


In [11]:
X_march = features_df[["avg_position", "impressions_total", "ctr", "word_count"]]
X_march_scaled = scaler_feb.transform(X_march)          # reuse Feb's scaler — do NOT refit
march_labels_oot = km_feb.predict(X_march_scaled)        # reuse Feb's cluster model — do NOT refit

features_df["cluster_oot"] = march_labels_oot
oot_profile = features_df.groupby("cluster_oot")[["avg_position","impressions_total","ctr","word_count"]].mean()
oot_profile["n_pages"] = features_df["cluster_oot"].value_counts().sort_index()

print("Feb-fit model applied OUT-OF-TIME to March data:")
print(oot_profile.round(3))

Feb-fit model applied OUT-OF-TIME to March data:
             avg_position  impressions_total    ctr  word_count  n_pages
cluster_oot                                                             
0                  11.412          26827.421  0.003    3000.319     4020
1                   9.847           1133.029  0.003    2735.927   139923
2                  50.301            444.340  0.001    2724.044    31069
3                   8.714              1.932  0.727    1423.274      292


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [12]:
X_no_ctr = features_df[["avg_position", "impressions_total", "word_count"]]
X_no_ctr_scaled = StandardScaler().fit_transform(X_no_ctr)
km_no_ctr = KMeans(n_clusters=4, random_state=42, n_init=10).fit(X_no_ctr_scaled)

X_scaled_march_full = StandardScaler().fit_transform(features_df[["avg_position", "impressions_total", "ctr", "word_count"]])
km_march_full = KMeans(n_clusters=4, random_state=42, n_init=10).fit(X_scaled_march_full)

sil_with_ctr = silhouette_score(X_scaled_march_full, km_march_full.labels_, sample_size=10000, random_state=42)
sil_without_ctr = silhouette_score(X_no_ctr_scaled, km_no_ctr.labels_, sample_size=10000, random_state=42)
print(f"Silhouette WITH ctr: {sil_with_ctr:.3f}")
print(f"Silhouette WITHOUT ctr: {sil_without_ctr:.3f}")

no_ctr_centers = pd.DataFrame(km_no_ctr.cluster_centers_, columns=["avg_position","impressions_total","word_count"])
print(no_ctr_centers.round(3))

Silhouette WITH ctr: 0.470
Silhouette WITHOUT ctr: 0.493
   avg_position  impressions_total  word_count
0        -0.388             -0.069      -0.251
1        -0.326              5.769       0.201
2         1.990             -0.232      -0.094
3        -0.073             -0.004       2.109


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.